# esg_passage_exp1

본 노트북은 기업 사업보고서 전체 텍스트를 활용하여 ESG 관련 표현을 정량화하고, 이를 실제 ESG 등급과 비교 및 검증하기 위한 분석 코드이다.

1) 주요 사용 데이터
 - `dart_full_texts.csv`: 기업·연도별 사업보고서 전체 텍스트
 - `seed_dictionary.csv`: E/S/G별 seed term 및 정규표현식
 - `company_master.csv`: 기업 정보 및 실제 ESG 등급

2)  주요 분석 흐름
 - 전체 보고서 텍스트 불러오기
 - 컬럼 형식 표준화
 - seed dictionary 불러오기
 - ESG passage 후보 추출
 - passage 중복 제거
 - 기업·연도 단위 document 생성
 - seed count score 계산
 - seed TF-IDF score 계산
 - FastText 모델 학습
 - seed term 확장 단어 생성
 - expanded count score 계산
 - ESG 등급 데이터 병합
 - Spearman 상관분석
 - t-test / Mann-Whitney 검정
 - OLS 회귀분석
 - Binary Logit 회귀분석
 - Ordered Logit 회귀분석
 - 최종 분석 데이터 저장


# 0. 라이브러리 및 경로 설정

In [3]:
# ============================================================
# 0. 라이브러리 및 경로 설정
# ============================================================

import os
import re
import warnings
from pathlib import Path
from html import unescape

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfTransformer
from gensim.models import FastText

from scipy.stats import spearmanr, ttest_ind, mannwhitneyu

import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel

warnings.filterwarnings("ignore")


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data").exists():
            return candidate
    return Path.cwd()


PROJECT_ROOT = find_project_root()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ANALYSIS_DIR = PROCESSED_DIR / "analysis"

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

COMPANY_MASTER_PATH = RAW_DIR / "company_master.csv"
SEED_DICT_PATH = RAW_DIR / "seed_dictionary.csv"

# 이번 분석의 핵심 원천 데이터
FULL_TEXTS_PATH = PROCESSED_DIR / "dart_full_texts.csv"

# 확인용 상태 파일
STATUS_PATH = PROCESSED_DIR / "dart_full_collection_status.csv"

# 분석 중간 산출물
ESG_PASSAGE_ANALYSIS_PATH = ANALYSIS_DIR / "dart_full_esg_passages_for_analysis.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("company_master:", COMPANY_MASTER_PATH)
print("seed_dictionary:", SEED_DICT_PATH)
print("dart_full_texts:", FULL_TEXTS_PATH)
print("status:", STATUS_PATH)
print("analysis output:", ANALYSIS_DIR)

PROJECT_ROOT: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final
company_master: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final\data\raw\company_master.csv
seed_dictionary: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final\data\raw\seed_dictionary.csv
dart_full_texts: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final\data\processed\dart_full_texts.csv
status: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final\data\processed\dart_full_collection_status.csv
analysis output: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final\data\processed\analysis


# 1. 전체 보고서 텍스트 데이터 불러오기

- 사용 파일: 'dart_full_texts.csv'
- 데이터 단위: 기업-연도별 전체 사업보고서 텍스트
- 목적: ESG passage 추출을 위한 원천 텍스트 확보
- 컬럼 표준화: 이후 병합/분석이 가능하도록 컬럼 형식 통일

In [4]:
# ============================================================
# 1. dart_full_texts.csv 불러오기
# ============================================================

if not FULL_TEXTS_PATH.exists():
    raise FileNotFoundError(f"dart_full_texts.csv 파일이 없습니다: {FULL_TEXTS_PATH}")

full_df = pd.read_csv(
    FULL_TEXTS_PATH,
    dtype={
        "stock_code": str,
        "corp_code": str,
        "rcept_no": str,
        "rcept_dt": str,
    }
)

full_df.columns = [str(c).strip() for c in full_df.columns]

print("전체 텍스트 행 수:", len(full_df))
print("컬럼:", full_df.columns.tolist())
print(full_df.head())

전체 텍스트 행 수: 381
컬럼: ['company_name', 'stock_code', 'corp_code', 'esg_year', 'report_fiscal_year', 'report_nm', 'rcept_no', 'rcept_dt', 'viewer_url', 'xml_file_names', 'full_text_chars', 'full_text']
  company_name stock_code corp_code  esg_year  report_fiscal_year  \
0         삼성전자     005930  00126380      2023                2022   
1         삼성전자     005930  00126380      2024                2023   
2         삼성전자     005930  00126380      2025                2024   
3          BYC     001460  00122579      2023                2022   
4          BYC     001460  00122579      2024                2023   

         report_nm        rcept_no  rcept_dt  \
0  사업보고서 (2022.12)  20230307000542  20230307   
1  사업보고서 (2023.12)  20240312000736  20240312   
2  사업보고서 (2024.12)  20250311001085  20250311   
3  사업보고서 (2022.12)  20230316001312  20230316   
4  사업보고서 (2023.12)  20240314001341  20240314   

                                          viewer_url  \
0  https://dart.fss.or.kr/dsaf001/main.do

In [5]:
# ============================================================
# 1-1. 컬럼 표준화
# ============================================================

# stock_code 정리
if "stock_code" not in full_df.columns:
    raise ValueError("dart_full_texts.csv에 stock_code 컬럼이 없습니다.")

full_df["stock_code"] = (
    full_df["stock_code"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(6)
)

# fiscal_year 정리
if "fiscal_year" not in full_df.columns:
    if "report_fiscal_year" in full_df.columns:
        full_df["fiscal_year"] = full_df["report_fiscal_year"]
    else:
        raise ValueError("dart_full_texts.csv에 fiscal_year 또는 report_fiscal_year 컬럼이 없습니다.")

full_df["fiscal_year"] = full_df["fiscal_year"].astype(int)

# esg_year 정리
if "esg_year" not in full_df.columns:
    full_df["esg_year"] = full_df["fiscal_year"] + 1
else:
    full_df["esg_year"] = full_df["esg_year"].astype(int)

# analysis_unit 생성
full_df["analysis_unit"] = (
    full_df["stock_code"].astype(str).str.zfill(6)
    + "x"
    + full_df["fiscal_year"].astype(str)
)

# 텍스트 컬럼 자동 탐색
text_col_candidates = [
    "full_text",
    "text",
    "unit_text",
    "report_text",
    "content",
    "paragraph",
    "passage",
    "section_text",
]

text_col = None

for c in text_col_candidates:
    if c in full_df.columns:
        text_col = c
        break

if text_col is None:
    raise ValueError(
        "텍스트 컬럼을 찾지 못했습니다. "
        "full_text, text, unit_text, report_text, content, paragraph, passage, section_text 중 하나가 필요합니다."
    )

print("사용할 텍스트 컬럼:", text_col)

# 선택적 메타 컬럼 없으면 생성
for col in ["company_name", "corp_code", "rcept_no", "rcept_dt", "viewer_url", "section_title"]:
    if col not in full_df.columns:
        full_df[col] = ""

if "section_order" not in full_df.columns:
    full_df["section_order"] = np.nan

if "section_level" not in full_df.columns:
    full_df["section_level"] = np.nan

if "section_tag" not in full_df.columns:
    full_df["section_tag"] = ""

print("분석 단위 수:", full_df["analysis_unit"].nunique())
print(full_df[["analysis_unit", "company_name", "stock_code", "fiscal_year", "esg_year", text_col]].head())

사용할 텍스트 컬럼: full_text
분석 단위 수: 381
  analysis_unit company_name stock_code  fiscal_year  esg_year  \
0   005930x2022         삼성전자     005930         2022      2023   
1   005930x2023         삼성전자     005930         2023      2024   
2   005930x2024         삼성전자     005930         2024      2025   
3   001460x2022          BYC     001460         2022      2023   
4   001460x2023          BYC     001460         2023      2024   

                                           full_text  
0  사업보고서 \n5.0 \n삼성전자주식회사 \n\nC \nA \n130111-0006...  
1  사업보고서 \n5.5 \n삼성전자주식회사 \n\nC \nA \nY \n130111-...  
2  연결감사보고서 \n6.0 \n삼성전자(주) \n\n00260116 \n1000000...  
3  사업보고서 \n5.0 \n(주)비와이씨 \n\nC \nA \n110111-00059...  
4  감사보고서 \n4.1 \n(주)비와이씨 \n\n00260116 \n100000000...  


# 2. seed dictionary 불러오기

- ESG 문장 추출 기준을 불러옴

In [6]:
# ============================================================
# 2. seed dictionary 불러오기
# ============================================================

seed_df = pd.read_csv(SEED_DICT_PATH)
seed_df.columns = [str(c).strip() for c in seed_df.columns]

required_cols = ["dimension", "seed_term", "pattern"]

for col in required_cols:
    if col not in seed_df.columns:
        raise ValueError(f"seed_dictionary.csv에 '{col}' 컬럼이 없습니다.")

valid_dims = {"E", "S", "G"}

if not set(seed_df["dimension"]).issubset(valid_dims):
    raise ValueError("dimension 컬럼에는 E, S, G만 들어가야 합니다.")

print("seed dictionary 행 수:", len(seed_df))
print(seed_df.head())

print("\n[dimension별 seed 개수]")
print(seed_df["dimension"].value_counts())

seed dictionary 행 수: 30
  dimension seed_term                 pattern  \
0         E        탄소                      탄소   
1         E      온실가스                온실가스|GHG   
2         E      탄소중립                    탄소중립   
3         E       넷제로   넷제로|net zero|net-zero   
4         E     재생에너지  재생에너지|renewable energy   

                                        source_basis  \
0  Bloomberg ESG climate change; Sautner-style cl...   
1  Bloomberg ESG climate change; GHG emissions li...   
2  Bloomberg ESG climate change; net-zero transit...   
3  Bloomberg ESG climate change; net-zero transit...   
4  Bloomberg ESG water/energy management; climate...   

                                       source_titles  \
0  Bloomberg ESG framework; Sautner-style climate...   
1  Bloomberg ESG framework; Sautner-style climate...   
2  Bloomberg ESG framework; Sautner-style climate...   
3  Bloomberg ESG framework; Sautner-style climate...   
4  Bloomberg ESG framework; Sautner-style climate...   

       

In [ ]:
# ============================================================
# 2-1. seed dictionary 검증
# ============================================================

required_cols = ["dimension", "seed_term", "pattern"]

for col in required_cols:
    if col not in seed_df.columns:
        raise ValueError(f"seed_dictionary.csv에 '{col}' 컬럼이 없습니다.")

valid_dims = {"E", "S", "G"}

if not set(seed_df["dimension"]).issubset(valid_dims):
    raise ValueError("dimension 컬럼에는 E, S, G만 들어가야 합니다.")

print("seed dictionary 검증 완료")

# 3. 전체 텍스트에서 ESG passage 후보 추출

- 전체 보고서 텍스트에서 ESG 관련 문장을 골라내는 단계
- 'seed dictionary'의 ESG 기준 단어와 정규표현식 사용

In [7]:
# ============================================================
# 3. 전체 텍스트에서 ESG passage 후보 추출
# ============================================================

def normalize_text(text):
    text = unescape(str(text))
    text = re.sub(r"[ \t\r\f]+", " ", text)
    text = re.sub(r"\n[ \t]*", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def split_passage_candidates(text):
    """
    dart_full_texts.csv의 텍스트 단위를 passage 후보로 분리한다.
    이미 문단 단위라면 거의 그대로 유지되고,
    긴 텍스트라면 문장/문단 단위로 나뉜다.
    """
    if pd.isna(text):
        return []

    text = normalize_text(text)

    chunks = re.split(
        r"\n{2,}|(?<=[다음임함음요오])\.\s+|(?<=[.!?。])\s+",
        text
    )

    passages = []

    for chunk in chunks:
        passage = normalize_text(chunk)

        if 40 <= len(passage) <= 2000:
            passages.append(passage)

    return passages


def count_pattern(text, pattern):
    try:
        return len(re.findall(str(pattern), str(text), flags=re.IGNORECASE))
    except re.error:
        return 0


def extract_esg_passages_from_full_texts(full_df, seed_df, text_col):
    rows = []
    seen = set()

    for _, row in full_df.iterrows():
        base = {
            "analysis_unit": row.get("analysis_unit", ""),
            "company_name": row.get("company_name", ""),
            "stock_code": row.get("stock_code", ""),
            "corp_code": row.get("corp_code", ""),
            "fiscal_year": row.get("fiscal_year", ""),
            "esg_year": row.get("esg_year", ""),
            "rcept_no": row.get("rcept_no", ""),
            "rcept_dt": row.get("rcept_dt", ""),
            "viewer_url": row.get("viewer_url", ""),
            "section_order": row.get("section_order", ""),
            "section_level": row.get("section_level", ""),
            "section_tag": row.get("section_tag", ""),
            "section_title": row.get("section_title", ""),
        }

        text = row.get(text_col, "")

        for sent_idx, passage in enumerate(split_passage_candidates(text)):
            matched_by_dim = {}
            matched_patterns_by_dim = {}

            for _, seed in seed_df.iterrows():
                dim = seed["dimension"]
                seed_term = str(seed["seed_term"])
                pattern = str(seed["pattern"])

                match_count = count_pattern(passage, pattern)

                if match_count > 0:
                    matched_by_dim.setdefault(dim, set()).add(seed_term)
                    matched_patterns_by_dim.setdefault(dim, set()).add(pattern)

            for dim, terms in matched_by_dim.items():
                key = (
                    base["analysis_unit"],
                    base["section_order"],
                    sent_idx,
                    dim,
                    passage[:300],
                )

                if key in seen:
                    continue

                seen.add(key)

                rows.append({
                    **base,
                    "sentence_index": sent_idx,
                    "passage": passage,
                    "dimension": dim,
                    "matched_terms": "|".join(sorted(terms)),
                    "matched_patterns": "|".join(sorted(matched_patterns_by_dim.get(dim, set()))),
                })

    return pd.DataFrame(rows)


df = extract_esg_passages_from_full_texts(full_df, seed_df, text_col)

print("추출된 ESG passage 행 수:", len(df))
print("분석 단위 수:", df["analysis_unit"].nunique() if len(df) > 0 else 0)
print(df.head())

df.to_csv(ESG_PASSAGE_ANALYSIS_PATH, index=False, encoding="utf-8-sig")
print("저장 완료:", ESG_PASSAGE_ANALYSIS_PATH)

추출된 ESG passage 행 수: 116610
분석 단위 수: 381
  analysis_unit company_name stock_code corp_code  fiscal_year  esg_year  \
0   005930x2022         삼성전자     005930  00126380         2022      2023   
1   005930x2022         삼성전자     005930  00126380         2022      2023   
2   005930x2022         삼성전자     005930  00126380         2022      2023   
3   005930x2022         삼성전자     005930  00126380         2022      2023   
4   005930x2022         삼성전자     005930  00126380         2022      2023   

         rcept_no  rcept_dt  \
0  20230307000542  20230307   
1  20230307000542  20230307   
2  20230307000542  20230307   
3  20230307000542  20230307   
4  20230307000542  20230307   

                                          viewer_url  section_order  \
0  https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...            NaN   
1  https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...            NaN   
2  https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...            NaN   
3  https://dart.fss.or.kr/dsaf0

# 4. passage 중복 정리

- ESG passage 후보 중 중복 행 제거
- 동일 기업-연도에서 같은 문장이 반복 추출되는 문제 방지
- 동일 passage가 여러 번 계산되어 점수가 과대평가되는 문제 방지

In [8]:
# ============================================================
# 4. passage 중복 정리
# ============================================================

df = df.drop_duplicates()

dedup_subset = [
    "analysis_unit",
    "stock_code",
    "fiscal_year",
    "rcept_no",
    "section_order",
    "sentence_index",
    "passage",
    "dimension",
]

dedup_subset = [c for c in dedup_subset if c in df.columns]

df_dedup = df.drop_duplicates(subset=dedup_subset).copy()

print("원본 ESG passage 행 수:", len(df))
print("중복 제거 후 행 수:", len(df_dedup))
print("분석 단위 수:", df_dedup["analysis_unit"].nunique())

원본 ESG passage 행 수: 116610
중복 제거 후 행 수: 116610
분석 단위 수: 381


# 5. 기업-연도 단위 document 만들기

- 같은 기업-연도의 ESG passage를 하나의 문서로 통합
- `document`: 해당 기업·연도의 ESG passage 전체를 이어붙인 텍스트
- `passage_count`: 해당 기업·연도에서 추출된 ESG passage 총개수
- `e_passage_count`: 환경(E) 관련 passage 수
- `s_passage_count`: 사회(S) 관련 passage 수
- `g_passage_count`: 지배구조(G) 관련 passage 수

In [9]:
# ============================================================
# 5. stock_code x fiscal_year 단위 document 생성
# ============================================================

doc_df = (
    df_dedup
    .groupby(
        ["analysis_unit", "stock_code", "corp_code", "company_name", "fiscal_year", "esg_year"],
        as_index=False
    )
    .agg(
        document=("passage", lambda x: " ".join(x.astype(str))),
        passage_count=("passage", "count"),
        e_passage_count=("dimension", lambda x: (x == "E").sum()),
        s_passage_count=("dimension", lambda x: (x == "S").sum()),
        g_passage_count=("dimension", lambda x: (x == "G").sum()),
    )
)

print("기업-연도 document 수:", len(doc_df))
print(doc_df.head())

기업-연도 document 수: 381
  analysis_unit stock_code corp_code company_name  fiscal_year  esg_year  \
0   000020x2022     000020  00119195         동화약품         2022      2023   
1   000020x2023     000020  00119195         동화약품         2023      2024   
2   000020x2024     000020  00119195         동화약품         2024      2025   
3   000040x2022     000040  00112378        KR모터스         2022      2023   
4   000040x2023     000040  00112378        KR모터스         2023      2024   

                                            document  passage_count  \
0  회사의 본점소재지 및 그 변경회사의 본점 소재지는 [서울시 중구 남대문로9길 24,...            160   
1  우리는 재무제표 감사와 관련된 대한민국의 윤리적 요구사항에 따라 회사로부터 독립적이...            159   
2  우리는 연결재무제표 감사와 관련된 대한민국의 윤리적 요구사항에 따라 회사로부터 독립...            154   
3  회사는 창원시 성산구 완암로 28에 위치하고 있으며,당기말 현재 주요주주는 엘브이엠...            118   
4  우리는 \n연결재무제표감사와 관련된 대한민국의 윤리적 요구사항에 따라 \n연결회사로...            135   

   e_passage_count  s_passage_count  g_passage_count  
0                0               24    

# 6. seed pattern 기반 출현 횟수 계산

- ESG 단어가 몇 번 나왔는지 계산

In [10]:
# ============================================================
# 6. seed pattern 기반 출현 횟수 계산
# ============================================================

seed_count_df = doc_df[
    ["analysis_unit", "stock_code", "corp_code", "company_name", "fiscal_year", "esg_year"]
].copy()

for _, row in seed_df.iterrows():
    dim = row["dimension"]
    seed_term = str(row["seed_term"])
    pattern = str(row["pattern"])

    safe_term = re.sub(r"[^가-힣a-zA-Z0-9_]+", "_", seed_term)
    col_name = f"seed_count_{dim}_{safe_term}"

    seed_count_df[col_name] = doc_df["document"].apply(
        lambda x: count_pattern(x, pattern)
    )

print(seed_count_df.head())

  analysis_unit stock_code corp_code company_name  fiscal_year  esg_year  \
0   000020x2022     000020  00119195         동화약품         2022      2023   
1   000020x2023     000020  00119195         동화약품         2023      2024   
2   000020x2024     000020  00119195         동화약품         2024      2025   
3   000040x2022     000040  00112378        KR모터스         2022      2023   
4   000040x2023     000040  00112378        KR모터스         2023      2024   

   seed_count_E_탄소  seed_count_E_온실가스  seed_count_E_탄소중립  seed_count_E_넷제로  \
0                0                  0                  0                 0   
1                0                  0                  0                 0   
2                0                  0                  0                 0   
3                0                  0                  0                 0   
4                0                  0                  0                 0   

   ...  seed_count_G_이사회  seed_count_G_사외이사  seed_count_G_감사위원회  \
0  ... 

# 7. E/S/G별 seed count score 만들기

- 기준 데이터: seed term별 출현 횟수
- 처리 내용: E/S/G 차원별 seed count 합산

In [11]:
# ============================================================
# 7. E/S/G별 seed count score 계산
# ============================================================

analysis_df = doc_df.copy()

seed_count_cols = [
    col for col in seed_count_df.columns
    if col.startswith("seed_count_")
]

analysis_df = pd.concat(
    [
        analysis_df.reset_index(drop=True),
        seed_count_df[seed_count_cols].reset_index(drop=True)
    ],
    axis=1
)

for dim in ["E", "S", "G"]:
    dim_cols = [
        col for col in analysis_df.columns
        if col.startswith(f"seed_count_{dim}_")
    ]
    
    analysis_df[f"seed_count_score_{dim}"] = analysis_df[dim_cols].sum(axis=1)

analysis_df["seed_count_score_total"] = (
    analysis_df["seed_count_score_E"]
    + analysis_df["seed_count_score_S"]
    + analysis_df["seed_count_score_G"]
)

print(
    analysis_df[
        [
            "company_name", "stock_code", "fiscal_year", "esg_year", "passage_count",
            "seed_count_score_E",
            "seed_count_score_S",
            "seed_count_score_G",
            "seed_count_score_total",
        ]
    ].head()
)

  company_name stock_code  fiscal_year  esg_year  passage_count  \
0         동화약품     000020         2022      2023            160   
1         동화약품     000020         2023      2024            159   
2         동화약품     000020         2024      2025            154   
3        KR모터스     000040         2022      2023            118   
4        KR모터스     000040         2023      2024            135   

   seed_count_score_E  seed_count_score_S  seed_count_score_G  \
0                   0                  27                 205   
1                   0                  26                 201   
2                   0                  27                 193   
3                  16                  11                 132   
4                  15                  15                 145   

   seed_count_score_total  
0                     232  
1                     227  
2                     220  
3                     159  
4                     175  


# 8. seed TF-IDF 점수 만들기

- TF: 특정 기업-연도 document 안에서 해당 seem term이 얼마나 자주 등장하는지
- IDF: 여러 기업에서 흔하게 등장하는 단어의 영향 축소
- 단순히 많이 나온 단어보다 해당 기업-연도에서 상대적으로 특징적인 ESG 단어를 반영

In [12]:
# ============================================================
# 8. seed TF-IDF 점수 계산
# ============================================================

count_matrix = analysis_df[seed_count_cols].values

tfidf_transformer = TfidfTransformer()
tfidf_matrix = tfidf_transformer.fit_transform(count_matrix)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=[
        col.replace("seed_count_", "seed_tfidf_")
        for col in seed_count_cols
    ]
)

analysis_df = pd.concat(
    [
        analysis_df.reset_index(drop=True),
        tfidf_df.reset_index(drop=True)
    ],
    axis=1
)

for dim in ["E", "S", "G"]:
    dim_cols = [
        col for col in analysis_df.columns
        if col.startswith(f"seed_tfidf_{dim}_")
    ]
    
    analysis_df[f"seed_tfidf_score_{dim}"] = analysis_df[dim_cols].sum(axis=1)

analysis_df["seed_tfidf_score_total"] = (
    analysis_df["seed_tfidf_score_E"]
    + analysis_df["seed_tfidf_score_S"]
    + analysis_df["seed_tfidf_score_G"]
)

print(
    analysis_df[
        [
            "company_name", "stock_code", "fiscal_year", "esg_year",
            "seed_tfidf_score_E",
            "seed_tfidf_score_S",
            "seed_tfidf_score_G",
            "seed_tfidf_score_total",
        ]
    ].head()
)

  company_name stock_code  fiscal_year  esg_year  seed_tfidf_score_E  \
0         동화약품     000020         2022      2023            0.000000   
1         동화약품     000020         2023      2024            0.000000   
2         동화약품     000020         2024      2025            0.000000   
3        KR모터스     000040         2022      2023            0.252064   
4        KR모터스     000040         2023      2024            0.212009   

   seed_tfidf_score_S  seed_tfidf_score_G  seed_tfidf_score_total  
0            0.290503            2.227020                2.517524  
1            0.266592            2.096705                2.363296  
2            0.317117            2.045616                2.362734  
3            0.145476            1.727590                2.125131  
4            0.177435            1.699345                2.088789  


# 9. FastText 학습

- 실제 보고서 문장에서 단어들이 어떤 맥락으로 쓰이는지 학습
- 기존 seed term과 의미가 비슷한 확장 단어 탐색

In [13]:
# ============================================================
# 9. FastText 학습용 토큰화
# ============================================================

def clean_text(text):
    text = str(text)
    text = re.sub(r"[^가-힣a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text):
    return clean_text(text).split()


sentences = df_dedup["passage"].apply(tokenize).tolist()

print("FastText 학습 문장 수:", len(sentences))
print(sentences[:2])

FastText 학습 문장 수: 116610
[['당사는', '1984년', '2월', '28일', '정기주주총회', '결의에', '의거하여', '상호를', '삼성전자공업주식회사에서', '삼성전자주식회사로', '변경하였습니다'], ['지분', '100', '인수2017', '07', '04', '세계', '최대', '규모', '평택', '반도체', '라인', '가동2017', '11', '01', '프린팅솔루션', '사업', '매각2018', '08', '30', 'QLED', '8K', 'TV', '퀀텀닷', '기술과', '8K', '해상도', '적용', 'TV', '출시2019', '04', '03', '갤럭시', 'S10', '5G', '5G', '스마트폰', '출시2019', '06', '04', '소비자', '라이프스타일', '맞춤형', '냉장고', '비스포크', 'BESPOKE', '출시2019', '09', '04', '엑시노스', '980', '5G', '모바일', '프로세서', '공개2019', '09', '06', '갤럭시', '폴드', '새로운', '폼팩터', '폴더블폰', '공개2020', '01', '29', '그랑데', 'AI', '인공지능', '기반', '세탁기', '건조기', '출시2020', '02', '04', 'AI', '차세대', '슈퍼컴퓨터용', '초고속', 'DRAM', '출시2020', '02', '11', '갤럭시', 'Z', '플립', '폴더블폰', '공개2020', '08', '30', '세계', '최대', '규모', '평택', '반도체', '2라인', '가동2021', '01', '07', 'Neo', 'QLED', 'TV', '공개2021', '05', '11', '비스포크', '홈', '글로벌', '시장', '확대', '선언2021', '06', '03', '전세계', '반도체', '업체', '최초', '전', '사업장', '탄소', '물', '폐기물', '저감', '인증2021', '08', '11', '갤

In [14]:
# ============================================================
# 9-1. FastText 모델 학습
# ============================================================

ft_model = FastText(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=10
)

print("FastText vocab size:", len(ft_model.wv))

FastText vocab size: 92319


# 10. seed term 기반 확장 단어 만들기

- FastText가 학습한 단어 유사도를 이용해 기존 ESG seed term을 확장하는 단계
- 예: `탄소중립`과 유사한 단어, `이사회`와 유사한 단어 등을 자동 탐색
- 유사도 기준: `threshold` 값 이상인 단어만 선택

In [15]:
# ============================================================
# 10. FastText 기반 seed 단어 확장
# ============================================================

def expand_terms_fasttext(model, seed_terms, topn=30, threshold=0.55):
    expanded = set(seed_terms)

    for term in seed_terms:
        term = str(term)
        
        if term not in model.wv:
            continue

        similar_words = model.wv.most_similar(term, topn=topn)

        for word, sim in similar_words:
            if sim >= threshold:
                expanded.add(word)

    return sorted(expanded)


threshold_list = [0.50, 0.55, 0.60]

expanded_dict_by_theta = {}

for theta in threshold_list:
    expanded_dict_by_theta[theta] = {}

    print(f"\n========== threshold θ = {theta} ==========")

    for dim in ["E", "S", "G"]:
        dim_seed_terms = seed_df.loc[
            seed_df["dimension"] == dim,
            "seed_term"
        ].astype(str).tolist()

        expanded_terms = expand_terms_fasttext(
            ft_model,
            dim_seed_terms,
            topn=30,
            threshold=theta
        )

        expanded_dict_by_theta[theta][dim] = expanded_terms

        print(f"\n[{dim}] 확장 단어 수:", len(expanded_terms))
        print(expanded_terms[:50])


========== threshold θ = 0.5 ==========

[E] 확장 단어 수: 308
['2040탄소중립을', '2050', '2050Net', '2050년', '2050탄소중립', '2050탄소중립에', 'ESG신재생에너지', 'ESG펀드의', 'ESG혁신기업대출', 'RE100등', 'SGC투자조합', 'SGC투자조합에', 'Zero기술', '감축분', '감축실현', '감축의', '건축폐기물', '관리폐기물은', '국가온실가스', '기후변화대응', '기후위기대응을', '넷제로', '넷제로은행연합', '대기관리대기오염물질', '대기방지시설에', '대기오염물질', '대기오염물질관리', '대기오염물질로는', '대기오염물질인', '대상기업이며온실가스', '대응온실가스', '디에스앤', '디에스엔', '디에스투자', '로드맵의', '및온실가스', '배출량', '배출량과', '배출량과는', '배출량당기', '배출량도', '배출량은1', '배출량이', '배출량인', '배출량임', '배출방지시설', '배출시설', '배출시설과', '배출시설변경신고', '배출시설에']

[S] 확장 단어 수: 307
['1사1산1하천', 'CP지원', 'IRA발효로', 'PJT협력사', 'PMCT', 'RBA임직원', '간호', '건강친화경영', '건설산업은', '검토임직원', '경력직원', '공공비영리', '공공세무', '공급망', '공급망과', '공급망과의', '공급망금융', '공급망에', '공급망에서의', '공급망의', '공급망이', '공급망플랫폼', '공헌', '관련임직원', '광고물', '교란과', '교육DX', '교육계', '교육및', '교육시간', '교육시행', '교육업무', '교육업종에', '교육연구', '교육용', '교육운영', '교육자료', '교육장', '교육주요', '교육지원', '교육진행', '교육참가확인서', '교육청별', '교육컨텐츠', '교육협력', '교육훈련', '교육훈련비', '교육훈련을', '국립공원', '국제사회']

[G] 확장 단어 

# 11. 확장 단어 저장

- 확장한 ESG 단어 목록을 threshold별, E/S/G별로 정리해 CSV 파일로 저장

In [16]:
# ============================================================
# 11. 확장 단어 저장
# ============================================================

rows = []

for theta in threshold_list:
    for dim in ["E", "S", "G"]:
        terms = expanded_dict_by_theta[theta][dim]

        for rank, term in enumerate(terms[:100], start=1):
            rows.append({
                "theta": theta,
                "dimension": dim,
                "rank": rank,
                "term": term,
            })

expanded_terms_df = pd.DataFrame(rows)

expanded_terms_path = ANALYSIS_DIR / "expanded_terms_by_theta.csv"

expanded_terms_df.to_csv(
    expanded_terms_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", expanded_terms_path)
print(expanded_terms_df.head(30))

저장 완료: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final\data\processed\analysis\expanded_terms_by_theta.csv
    theta dimension  rank        term
0     0.5         E     1   2040탄소중립을
1     0.5         E     2        2050
2     0.5         E     3     2050Net
3     0.5         E     4       2050년
4     0.5         E     5    2050탄소중립
5     0.5         E     6   2050탄소중립에
6     0.5         E     7   ESG신재생에너지
7     0.5         E     8      ESG펀드의
8     0.5         E     9   ESG혁신기업대출
9     0.5         E    10      RE100등
10    0.5         E    11     SGC투자조합
11    0.5         E    12    SGC투자조합에
12    0.5         E    13      Zero기술
13    0.5         E    14         감축분
14    0.5         E    15        감축실현
15    0.5         E    16         감축의
16    0.5         E    17       건축폐기물
17    0.5         E    18      관리폐기물은
18    0.5         E    19      국가온실가스
19    0.5         E    20      기후변화대응
20    0.5         E    21     기후위기대응을
21    0.5         E    22         넷제로
22    0.5

# 12. expanded count score 계산

- 확장 단어가 문서 안에 몇 번 등장하는지 계산

In [17]:
# ============================================================
# 12. expanded score 계산 함수
# ============================================================

def make_expanded_score(doc_df, expanded_terms_dict):
    result = doc_df.copy()

    for dim in ["E", "S", "G"]:
        terms = expanded_terms_dict[dim]
        temp_score = pd.Series(0, index=result.index)

        for term in terms:
            pattern = re.escape(str(term))
            temp_score += result["document"].apply(
                lambda x: count_pattern(x, pattern)
            )

        result[f"expanded_count_score_{dim}"] = temp_score

    result["expanded_count_score_total"] = (
        result["expanded_count_score_E"]
        + result["expanded_count_score_S"]
        + result["expanded_count_score_G"]
    )

    return result

In [18]:
# ============================================================
# 12-1. threshold별 expanded score 만들기
# ============================================================

expanded_results = {}

for theta in threshold_list:
    temp_df = make_expanded_score(
        doc_df,
        expanded_dict_by_theta[theta]
    )

    expanded_results[theta] = temp_df

    print(f"\nθ = {theta}")
    print(
        temp_df[
            [
                "company_name", "stock_code", "fiscal_year", "esg_year",
                "expanded_count_score_E",
                "expanded_count_score_S",
                "expanded_count_score_G",
                "expanded_count_score_total",
            ]
        ].head()
    )


θ = 0.5
  company_name stock_code  fiscal_year  esg_year  expanded_count_score_E  \
0         동화약품     000020         2022      2023                       0   
1         동화약품     000020         2023      2024                       0   
2         동화약품     000020         2024      2025                       0   
3        KR모터스     000040         2022      2023                      22   
4        KR모터스     000040         2023      2024                      22   

   expanded_count_score_S  expanded_count_score_G  expanded_count_score_total  
0                      35                     220                         255  
1                      35                     214                         249  
2                      34                     205                         239  
3                      10                     138                         170  
4                      13                     151                         186  

θ = 0.55
  company_name stock_code  fiscal_year  esg_

# 13. ESG 등급 데이터 불러오기

- 사용 파일: `company_master.csv`
- 처리 내용: 종목코드 6자리 문자열 / 사업연도, ESG 평가연도 형식 정리 / ESG 등급을 숫자값으로 변환
- 목적: 텍스트 기반 ESG 점수와 실제 ESG 등급 비교 준비

In [19]:
# ============================================================
# 13. ESG 등급 데이터 불러오기
# ============================================================

grade_df = pd.read_csv(
    COMPANY_MASTER_PATH,
    dtype={"stock_code": str}
)

grade_df.columns = [str(c).strip() for c in grade_df.columns]

grade_df["stock_code"] = (
    grade_df["stock_code"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(6)
)

grade_df["fiscal_year"] = grade_df["fiscal_year"].astype(int)

if "esg_year" in grade_df.columns:
    grade_df["esg_year"] = grade_df["esg_year"].astype(int)
else:
    grade_df["esg_year"] = grade_df["fiscal_year"] + 1

grade_map = {
    "D": 0,
    "C": 1,
    "B": 2,
    "B+": 3,
    "A": 4,
    "A+": 5,
    "S": 6,
}

if "esg_grade_num" not in grade_df.columns:
    grade_df["esg_grade"] = grade_df["esg_grade"].astype(str).str.strip()
    grade_df["esg_grade_num"] = grade_df["esg_grade"].map(grade_map)

print("ESG 등급 데이터 수:", len(grade_df))
print(grade_df[["company_name", "stock_code", "fiscal_year", "esg_year", "esg_grade", "esg_grade_num"]].head())

print("\n[ESG 등급 분포]")
print(grade_df["esg_grade"].value_counts(dropna=False))

ESG 등급 데이터 수: 381
  company_name stock_code  fiscal_year  esg_year esg_grade  esg_grade_num
0         삼성전자     005930         2022      2023         A              4
1         삼성전자     005930         2023      2024        B+              3
2         삼성전자     005930         2024      2025         A              4
3          BYC     001460         2022      2023         D              0
4          BYC     001460         2023      2024         D              0

[ESG 등급 분포]
esg_grade
A     123
B+     76
D      64
C      53
B      36
A+     29
Name: count, dtype: int64


# 14. seed score와 ESG 등급 병합

- 텍스트 기반 ESG 점수와 실제 ESG 등급을 같은 분석 단위로 연결

In [20]:
# ============================================================
# 14. seed score + ESG 등급 병합
# ============================================================

analysis_df["stock_code"] = analysis_df["stock_code"].astype(str).str.zfill(6)
analysis_df["fiscal_year"] = analysis_df["fiscal_year"].astype(int)
analysis_df["esg_year"] = analysis_df["esg_year"].astype(int)

grade_df["stock_code"] = grade_df["stock_code"].astype(str).str.zfill(6)
grade_df["fiscal_year"] = grade_df["fiscal_year"].astype(int)
grade_df["esg_year"] = grade_df["esg_year"].astype(int)

grade_cols = [
    "stock_code",
    "fiscal_year",
    "esg_year",
    "esg_grade",
    "esg_grade_num",
]

for c in ["e_grade", "s_grade", "g_grade", "esg_source"]:
    if c in grade_df.columns:
        grade_cols.append(c)

analysis_seed = pd.merge(
    analysis_df,
    grade_df[grade_cols],
    on=["stock_code", "fiscal_year", "esg_year"],
    how="inner"
)

print("분석 데이터 수:", len(analysis_seed))
print(
    analysis_seed[
        [
            "company_name", "stock_code", "fiscal_year", "esg_year",
            "esg_grade", "esg_grade_num",
            "seed_count_score_E",
            "seed_count_score_S",
            "seed_count_score_G",
            "seed_tfidf_score_E",
            "seed_tfidf_score_S",
            "seed_tfidf_score_G",
        ]
    ].head()
)

분석 데이터 수: 381
  company_name stock_code  fiscal_year  esg_year esg_grade  esg_grade_num  \
0         동화약품     000020         2022      2023         C              1   
1         동화약품     000020         2023      2024         C              1   
2         동화약품     000020         2024      2025         C              1   
3        KR모터스     000040         2022      2023         D              0   
4        KR모터스     000040         2023      2024         D              0   

   seed_count_score_E  seed_count_score_S  seed_count_score_G  \
0                   0                  27                 205   
1                   0                  26                 201   
2                   0                  27                 193   
3                  16                  11                 132   
4                  15                  15                 145   

   seed_tfidf_score_E  seed_tfidf_score_S  seed_tfidf_score_G  
0            0.000000            0.290503            2.227020  
1   

# 15. Spearman 순위 상관 분석

- 상관계수 `+`: ESG 점수가 높을수록 등급도 높은 경향
- 상관계수 `-`: ESG 점수가 높을수록 등급은 낮은 경향
- `p-value < 0.05`: 통계적으로 유의

In [21]:
# ============================================================
# 15. Spearman 순위 상관 분석
# ============================================================

feature_cols = [
    "passage_count",
    "e_passage_count",
    "s_passage_count",
    "g_passage_count",
    "seed_count_score_E",
    "seed_count_score_S",
    "seed_count_score_G",
    "seed_count_score_total",
    "seed_tfidf_score_E",
    "seed_tfidf_score_S",
    "seed_tfidf_score_G",
    "seed_tfidf_score_total",
]

spearman_rows = []

for col in feature_cols:
    corr, pval = spearmanr(
        analysis_seed[col],
        analysis_seed["esg_grade_num"]
    )

    spearman_rows.append({
        "feature": col,
        "spearman_corr": corr,
        "p_value": pval,
    })

spearman_result_df = pd.DataFrame(spearman_rows)
spearman_result_df = spearman_result_df.sort_values(
    "spearman_corr",
    ascending=False
)

print(spearman_result_df)

spearman_path = ANALYSIS_DIR / "seed_spearman_result.csv"

spearman_result_df.to_csv(
    spearman_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", spearman_path)

                   feature  spearman_corr       p_value
7   seed_count_score_total       0.732722  2.483277e-65
0            passage_count       0.720943  2.454521e-62
6       seed_count_score_G       0.711408  5.053531e-60
5       seed_count_score_S       0.699500  2.909146e-57
2          s_passage_count       0.694447  3.927283e-56
3          g_passage_count       0.694336  4.157280e-56
11  seed_tfidf_score_total       0.647312  1.285761e-46
9       seed_tfidf_score_S       0.511875  7.849729e-27
4       seed_count_score_E       0.507264  2.636963e-26
1          e_passage_count       0.498763  2.347589e-25
8       seed_tfidf_score_E       0.356810  6.999328e-13
10      seed_tfidf_score_G       0.206441  4.904201e-05
저장 완료: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final\data\processed\analysis\seed_spearman_result.csv


# 16. expanded score도 ESG 등급과 비교

- FastText로 확장한 ESG 단어 점수와 실제 ESG 등급의 관계를 threshold별로 비교

In [22]:
# ============================================================
# 16. threshold별 expanded score Spearman 검증
# ============================================================

expanded_eval_rows = []

for theta, temp_df in expanded_results.items():
    temp_df = temp_df.copy()
    temp_df["stock_code"] = temp_df["stock_code"].astype(str).str.zfill(6)
    temp_df["fiscal_year"] = temp_df["fiscal_year"].astype(int)
    temp_df["esg_year"] = temp_df["esg_year"].astype(int)

    temp_analysis = pd.merge(
        temp_df,
        grade_df[grade_cols],
        on=["stock_code", "fiscal_year", "esg_year"],
        how="inner"
    )

    expanded_features = [
        "expanded_count_score_E",
        "expanded_count_score_S",
        "expanded_count_score_G",
        "expanded_count_score_total",
    ]

    for col in expanded_features:
        corr, pval = spearmanr(
            temp_analysis[col],
            temp_analysis["esg_grade_num"]
        )

        expanded_eval_rows.append({
            "theta": theta,
            "feature": col,
            "spearman_corr": corr,
            "p_value": pval,
        })

expanded_eval_df = pd.DataFrame(expanded_eval_rows)
expanded_eval_df = expanded_eval_df.sort_values(
    "spearman_corr",
    ascending=False
)

print(expanded_eval_df)

expanded_spearman_path = ANALYSIS_DIR / "expanded_spearman_result.csv"

expanded_eval_df.to_csv(
    expanded_spearman_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", expanded_spearman_path)

    theta                     feature  spearman_corr       p_value
3    0.50  expanded_count_score_total       0.712940  2.178820e-60
7    0.55  expanded_count_score_total       0.712940  2.178820e-60
11   0.60  expanded_count_score_total       0.712924  2.197903e-60
2    0.50      expanded_count_score_G       0.706181  8.551847e-59
10   0.60      expanded_count_score_G       0.706181  8.551847e-59
6    0.55      expanded_count_score_G       0.706181  8.551847e-59
5    0.55      expanded_count_score_S       0.658598  9.708501e-49
1    0.50      expanded_count_score_S       0.658598  9.708501e-49
9    0.60      expanded_count_score_S       0.658596  9.717771e-49
0    0.50      expanded_count_score_E       0.480761  1.975383e-23
4    0.55      expanded_count_score_E       0.480761  1.975383e-23
8    0.60      expanded_count_score_E       0.480761  1.975383e-23
저장 완료: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final\data\processed\analysis\expanded_spearman_result.csv


# 17. t-test/Mann-Whitney 검정

- 그룹 기준:
  - `high_esg = 1`: ESG 우수 그룹
  - `high_esg = 0`: ESG 비우수 그룹
- 우수 그룹 기준: `esg_grade_num >= 5`
- t-test: 두 그룹의 평균 차이 검정
- Mann-Whitney 검정: 두 그룹의 점수 분포 차이 검정
- 해석 기준:
  - 우수 그룹 평균 > 비우수 그룹 평균
  - `p-value < 0.05`이면 통계적으로 유의

In [23]:
# ============================================================
# 17. ESG 우수 그룹 생성
# ============================================================

analysis_seed["high_esg"] = np.where(
    analysis_seed["esg_grade_num"] >= 5,
    1,
    0
)

print(analysis_seed["high_esg"].value_counts())

high_esg
0    352
1     29
Name: count, dtype: int64


In [24]:
# ============================================================
# 17-1. t-test / Mann-Whitney 검정
# ============================================================

test_features = [
    "seed_count_score_E",
    "seed_count_score_S",
    "seed_count_score_G",
    "seed_count_score_total",
    "seed_tfidf_score_E",
    "seed_tfidf_score_S",
    "seed_tfidf_score_G",
    "seed_tfidf_score_total",
]

test_rows = []

for col in test_features:
    high = analysis_seed.loc[analysis_seed["high_esg"] == 1, col]
    low = analysis_seed.loc[analysis_seed["high_esg"] == 0, col]

    t_stat, t_p = ttest_ind(high, low, equal_var=False)
    u_stat, u_p = mannwhitneyu(high, low, alternative="greater")

    test_rows.append({
        "feature": col,
        "high_mean": high.mean(),
        "low_mean": low.mean(),
        "t_test_p_value": t_p,
        "mann_whitney_p_value": u_p,
    })

test_result_df = pd.DataFrame(test_rows)

print(test_result_df)

group_test_path = ANALYSIS_DIR / "group_test_result.csv"

test_result_df.to_csv(
    group_test_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", group_test_path)

                  feature   high_mean    low_mean  t_test_p_value  \
0      seed_count_score_E  172.862069   75.460227        0.026387   
1      seed_count_score_S  166.482759   77.127841        0.005762   
2      seed_count_score_G  565.517241  361.468750        0.000029   
3  seed_count_score_total  904.862069  514.056818        0.000392   
4      seed_tfidf_score_E    0.648051    0.397120        0.003798   
5      seed_tfidf_score_S    0.515544    0.383812        0.008103   
6      seed_tfidf_score_G    1.868497    1.907335        0.489371   
7  seed_tfidf_score_total    3.032091    2.688268        0.000014   

   mann_whitney_p_value  
0          2.359967e-06  
1          1.473216e-06  
2          4.694989e-07  
3          2.360491e-07  
4          1.331257e-04  
5          1.301300e-03  
6          7.974045e-01  
7          1.222485e-05  
저장 완료: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final\data\processed\analysis\group_test_result.csv


# 18. OLS 회귀분석

- OLS 선형회귀분석: 다른 변수들을 함께 고려했을 때, 각 텍스트 점수가 ESG 등급과 어떤 관계를 가지는지 확인
- 해석 기준:
  - `coef > 0`: 해당 점수가 높을수록 ESG 등급도 높아지는 경향
  - `coef < 0`: 해당 점수가 높을수록 ESG 등급은 낮아지는 경향
  - `p-value < 0.05`: 통계적으로 유의

In [25]:
# ============================================================
# 18. OLS 회귀분석
# ============================================================

ols_features = [
    "passage_count",
    "seed_tfidf_score_E",
    "seed_tfidf_score_S",
    "seed_tfidf_score_G",
]

ols_data = analysis_seed.dropna(
    subset=ols_features + ["esg_grade_num"]
).copy()

X = ols_data[ols_features]
X = sm.add_constant(X)

y = ols_data["esg_grade_num"]

ols_model = sm.OLS(y, X).fit()

print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:          esg_grade_num   R-squared:                       0.503
Model:                            OLS   Adj. R-squared:                  0.498
Method:                 Least Squares   F-statistic:                     95.25
Date:                Sat, 23 May 2026   Prob (F-statistic):           7.04e-56
Time:                        01:10:44   Log-Likelihood:                -591.43
No. Observations:                 381   AIC:                             1193.
Df Residuals:                     376   BIC:                             1213.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -5.1598      0

# 19. Binary Logit 회귀분석

- Binary Logit 회귀분석: 텍스트 점수가 높을수록 ESG 우수 그룹에 속할 가능성이 증가하는지 확인
- 해석 기준:
  - `coef > 0`: 해당 변수가 높을수록 ESG 우수 그룹일 가능성 증가
  - `coef < 0`: 해당 변수가 높을수록 ESG 우수 그룹일 가능성 감소
  - `p-value < 0.05`: 통계적으로 유의

In [26]:
# ============================================================
# 19. Binary Logit 회귀분석
# ============================================================

logit_features = [
    "passage_count",
    "seed_tfidf_score_E",
    "seed_tfidf_score_S",
    "seed_tfidf_score_G",
]

logit_data = analysis_seed.dropna(
    subset=logit_features + ["high_esg"]
).copy()

X = logit_data[logit_features]
X = sm.add_constant(X)

y = logit_data["high_esg"]

logit_model = sm.Logit(y, X).fit()

print(logit_model.summary())

Optimization terminated successfully.
         Current function value: 0.240598
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:               high_esg   No. Observations:                  381
Model:                          Logit   Df Residuals:                      376
Method:                           MLE   Df Model:                            4
Date:                Sat, 23 May 2026   Pseudo R-squ.:                  0.1062
Time:                        01:10:47   Log-Likelihood:                -91.668
converged:                       True   LL-Null:                       -102.56
Covariance Type:            nonrobust   LLR p-value:                 0.0002219
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -8.5465      2.213     -3.861      0.000     -12.885      -4.208
passa

# 20. Ordered Logit 회귀분석

- ESG 등급의 순서형 특성을 반영한 회귀분석
- Ordered Logit 회귀분석: 텍스트 점수가 높을수록 더 높은 ESG 등급에 속할 가능성이 커지는지 확인
- OLS와 차이:
  - OLS: ESG 등급을 연속형 숫자처럼 처리
  - Ordered Logit: ESG 등급을 순서가 있는 범주형 변수로 처리

In [27]:
# ============================================================
# 20. Ordered Logit 회귀분석
# ============================================================

ordered_features = [
    "passage_count",
    "seed_tfidf_score_E",
    "seed_tfidf_score_S",
    "seed_tfidf_score_G",
]

ordered_data = analysis_seed.dropna(
    subset=ordered_features + ["esg_grade_num"]
).copy()

X = ordered_data[ordered_features]
y = ordered_data["esg_grade_num"]

ordered_model = OrderedModel(
    y,
    X,
    distr="logit"
)

ordered_result = ordered_model.fit(method="bfgs")

print(ordered_result.summary())

Optimization terminated successfully.
         Current function value: 1.348877
         Iterations: 41
         Function evaluations: 45
         Gradient evaluations: 45
                             OrderedModel Results                             
Dep. Variable:          esg_grade_num   Log-Likelihood:                -513.92
Model:                   OrderedModel   AIC:                             1046.
Method:            Maximum Likelihood   BIC:                             1081.
Date:                Sat, 23 May 2026                                         
Time:                        01:10:48                                         
No. Observations:                 381                                         
Df Residuals:                     372                                         
Df Model:                           4                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------

# 21. 최종 분석 데이터 저장

- 저장 파일: `final_analysis_seed_score.csv`
- 포함 내용:
  - 기업 정보
  - 사업연도·ESG 평가연도
  - passage 수
  - seed count score
  - seed TF-IDF score
  - ESG 등급
  - ESG 등급 숫자값

In [28]:
# ============================================================
# 21. 최종 분석 데이터 저장
# ============================================================

final_analysis_path = ANALYSIS_DIR / "final_analysis_seed_score.csv"

analysis_seed.to_csv(
    final_analysis_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", final_analysis_path)
print("최종 분석 데이터 수:", len(analysis_seed))
print(analysis_seed.head())

저장 완료: c:\Users\duswl\OneDrive\바탕 화면\duswls3247\비정형 데이터 처리\Final\data\processed\analysis\final_analysis_seed_score.csv
최종 분석 데이터 수: 381
  analysis_unit stock_code corp_code company_name  fiscal_year  esg_year  \
0   000020x2022     000020  00119195         동화약품         2022      2023   
1   000020x2023     000020  00119195         동화약품         2023      2024   
2   000020x2024     000020  00119195         동화약품         2024      2025   
3   000040x2022     000040  00112378        KR모터스         2022      2023   
4   000040x2023     000040  00112378        KR모터스         2023      2024   

                                            document  passage_count  \
0  회사의 본점소재지 및 그 변경회사의 본점 소재지는 [서울시 중구 남대문로9길 24,...            160   
1  우리는 재무제표 감사와 관련된 대한민국의 윤리적 요구사항에 따라 회사로부터 독립적이...            159   
2  우리는 연결재무제표 감사와 관련된 대한민국의 윤리적 요구사항에 따라 회사로부터 독립...            154   
3  회사는 창원시 성산구 완암로 28에 위치하고 있으며,당기말 현재 주요주주는 엘브이엠...            118   
4  우리는 \n연결재무제표감사와 관련된 대한민국의 윤리적 요구사항에 따라 \n연결회사로...